In [7]:
import sys
import os
import math
import numpy as np



In [8]:
states = {
    "s": 0,
    "E": 1,
    "5": 2,
    "I": 3,
    "e": 4
}

id2state = {
    0: "s",
    1: "E",
    2: "5",
    3: "I",
    4: "e"
}


In [9]:
state_transition_prob = np.array([
    [0.0, 1.0, 0.0, 0.0, 0.0],
    [0.0, 0.9, 0.1, 0.0, 0.0],
    [0.0, 0.0, 0.0, 1.0, 0.0],
    [0.0, 0.0, 0.0, 0.9, 0.1],
    [0.0, 0.0, 0.0, 0.0, 0.0]
])

In [10]:
emission_nuc_codes = {
    'A': 0,
    'C': 1,
    'G': 2,
    'T': 3
}

emission_probs = np.array([
    [0.00, 0.00, 0.00, 0.00],
    [0.25, 0.25, 0.25, 0.25],
    [0.05, 0.00, 0.95, 0.00],
    [0.40, 0.10, 0.10, 0.40],
    [0.00, 0.00, 0.00, 0.00]
])

In [11]:
query_sequence = "CTTCATGTGAAAGCAGACGTAAGTCA"

In [13]:
num_states = len(states)

sequence_length = len(query_sequence)

viterbi_value_matrix = np.full(
    (num_states, sequence_length),
    -np.inf
)

viterbi_trace_matrix = np.zeros(
    (num_states, sequence_length),
    dtype=int
)

In [14]:
first_nucleotide = emission_nuc_codes[
    query_sequence[0]
]

for state in range(num_states):

    transition_prob = state_transition_prob[
        states["s"]
    ][state]

    emission_prob = emission_probs[
        state
    ][first_nucleotide]

    if transition_prob > 0 and emission_prob > 0:

        viterbi_value_matrix[state][0] = (
            math.log(transition_prob)
            + math.log(emission_prob)
        )

In [15]:
print(viterbi_value_matrix)

[[       -inf        -inf        -inf        -inf        -inf        -inf
         -inf        -inf        -inf        -inf        -inf        -inf
         -inf        -inf        -inf        -inf        -inf        -inf
         -inf        -inf        -inf        -inf        -inf        -inf
         -inf        -inf]
 [-1.38629436        -inf        -inf        -inf        -inf        -inf
         -inf        -inf        -inf        -inf        -inf        -inf
         -inf        -inf        -inf        -inf        -inf        -inf
         -inf        -inf        -inf        -inf        -inf        -inf
         -inf        -inf]
 [       -inf        -inf        -inf        -inf        -inf        -inf
         -inf        -inf        -inf        -inf        -inf        -inf
         -inf        -inf        -inf        -inf        -inf        -inf
         -inf        -inf        -inf        -inf        -inf        -inf
         -inf        -inf]
 [       -inf        -inf      

In [16]:
def calculate_prob_for_a_node(current_state, col):

    max_prob = -np.inf

    best_previous_state = 0

    nucleotide = query_sequence[col]

    nucleotide_code = emission_nuc_codes[nucleotide]

    emission_prob = emission_probs[
        current_state
    ][nucleotide_code]

    if emission_prob == 0:
        return max_prob, best_previous_state

    for previous_state in range(num_states):

        transition_prob = state_transition_prob[
            previous_state
        ][current_state]

        if transition_prob == 0:
            continue

        previous_prob = viterbi_value_matrix[
            previous_state
        ][col - 1]

        if previous_prob == -np.inf:
            continue

        current_prob = (
            previous_prob
            + math.log(transition_prob)
            + math.log(emission_prob)
        )

        if current_prob > max_prob:

            max_prob = current_prob

            best_previous_state = previous_state

    return max_prob, best_previous_state

In [17]:
for col in range(1, sequence_length):

    for current_state in range(num_states):

        max_prob, best_previous_state = (
            calculate_prob_for_a_node(
                current_state,
                col
            )
        )

        viterbi_value_matrix[
            current_state
        ][col] = max_prob

        viterbi_trace_matrix[
            current_state
        ][col] = best_previous_state

In [18]:
print(viterbi_value_matrix)

[[        -inf         -inf         -inf         -inf         -inf
          -inf         -inf         -inf         -inf         -inf
          -inf         -inf         -inf         -inf         -inf
          -inf         -inf         -inf         -inf         -inf
          -inf         -inf         -inf         -inf         -inf
          -inf]
 [ -1.38629436  -2.87794924  -4.36960411  -5.86125899  -7.35291387
   -8.84456875 -10.33622362 -11.8278785  -13.31953338 -14.81118825
  -16.30284313 -17.79449801 -19.28615288 -20.77780776 -22.26946264
  -23.76111751 -25.25277239 -26.74442727 -28.23608214 -29.72773702
  -31.2193919  -32.71104677 -34.20270165 -35.69435653 -37.1860114
  -38.67766628]
 [        -inf         -inf         -inf         -inf -11.15957636
          -inf -11.19844713         -inf -14.18175689 -18.61785074
  -20.10950562 -21.6011605  -20.14837639         -inf -26.07612513
  -24.62334102 -29.05943488         -inf -29.09830565         -inf
  -35.02605439 -36.51770926 -35

In [19]:
print(viterbi_trace_matrix)

[[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
 [0 0 0 0 1 0 1 0 1 1 1 1 1 0 1 1 1 0 1 0 1 1 1 0 0 1]
 [0 0 0 0 0 2 3 2 3 2 3 3 3 3 3 3 2 3 3 2 3 3 3 3 3 3]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]]


In [20]:
def traceback():

    final_column = sequence_length - 1

    best_last_state = np.argmax(
        viterbi_value_matrix[:, final_column]
    )

    best_path = [best_last_state]

    for col in range(
        final_column,
        0,
        -1
    ):

        best_last_state = (
            viterbi_trace_matrix[
                best_last_state
            ][col]
        )

        best_path.insert(
            0,
            best_last_state
        )

    decoded_states = []

    for state_index in best_path:

        decoded_states.append(
            id2state[state_index]
        )

    return decoded_states

In [21]:
predicted_path = traceback()

print("DNA Sequence:")
print(query_sequence)

print("\nPredicted Hidden States:")
print("".join(predicted_path))

DNA Sequence:
CTTCATGTGAAAGCAGACGTAAGTCA

Predicted Hidden States:
EEEEEEEEEEEEEEEEEEEEEEEEEE
